# Vertex box exploration

The neutrino vertex is always at **time ≈ 0** (top of the image, row → 0)
and **wire ≈ 250** (centre of the wire axis, col ≈ 250).

Strategy:
1. Define a **search region** around (0, 250) — a coarse window that is guaranteed to contain the vertex.
2. Inside the search region, find the **local density maximum**: the pixel with the most active neighbours within radius *r*.
3. Place the final **evaluation box** centred on that refined vertex.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
%matplotlib inline

# ── Change this path to your .npz ──────────────────────────────────────────
NPZ_PATH = Path("../../dino_checkpoints/test_cropping/features_ep10.npz")
# ───────────────────────────────────────────────────────────────────────────

data      = np.load(NPZ_PATH)
positions = data["positions"]   # [N_valid, 2]  (row, col)
offsets   = data["offsets"]     # [N_images+1]
labels    = data["labels"]      # [N_images]

CLASS_NAMES = ["numu", "nue", "nutau", "NC"]
print(f"Images: {len(labels)}   Valid pixels: {positions.shape[0]}")
print({ CLASS_NAMES[c]: int((labels==c).sum()) for c in np.unique(labels) })

## Parameters

In [ ]:
# ── Coarse search region around the known vertex neighbourhood ──────────────
SEARCH_ROW  = 0     # search rows [SEARCH_ROW, SEARCH_ROW + SEARCH_H)
SEARCH_COL  = 250   # search cols [SEARCH_COL - SEARCH_W, SEARCH_COL + SEARCH_W)
SEARCH_H    = 80    # how far down from row 0 to look
SEARCH_W    = 100    # half-width around col 250

# ── Local density radius (pixels) ──────────────────────────────────────────
DENSITY_R   = 10.0

# ── Final evaluation box (centred on the refined vertex) ───────────────────
BOX_H = 30   # half-height
BOX_W = 30   # half-width

## Vertex finder

In [ ]:
def find_vertex(pos: np.ndarray,
                search_row: int, search_col: int,
                search_h: int, search_w: int,
                density_r: float) -> tuple[float, float]:
    """
    1. Restrict to active pixels inside the search region.
    2. Among those, return the one with the highest count of active neighbours
       (from the full image) within `density_r` pixels.
    Falls back to the search-region centroid if no pixel is found.
    """
    rows, cols = pos[:, 0], pos[:, 1]
    in_search = (
        (rows >= search_row) & (rows < search_row + search_h) &
        (cols >= search_col - search_w) & (cols < search_col + search_w)
    )
    candidates = pos[in_search]    # pixels inside the search region

    if len(candidates) == 0:
        return float(search_row), float(search_col)   # fallback

    # Count neighbours in the *full* image within radius density_r
    # candidates: [C, 2],  pos: [N, 2]
    diff   = candidates[:, None, :] - pos[None, :, :]   # [C, N, 2]
    counts = (np.linalg.norm(diff, axis=-1) < density_r).sum(axis=1)  # [C]

    best = candidates[counts.argmax()]
    return float(best[0]), float(best[1])

## Visual check — fixed search region vs refined vertex

In [ ]:
N_SHOW = 5
rng    = np.random.default_rng(0)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_imgs = np.where(labels == cls_idx)[0]
    if len(cls_imgs) == 0:
        continue
    chosen = rng.choice(cls_imgs, size=min(N_SHOW, len(cls_imgs)), replace=False)

    fig, axes = plt.subplots(1, len(chosen), figsize=(5 * len(chosen), 5), squeeze=False)

    for ax, img_idx in zip(axes[0], chosen):
        sl   = slice(offsets[img_idx], offsets[img_idx + 1])
        pos  = positions[sl].astype(np.float32)
        rows, cols = pos[:, 0], pos[:, 1]

        # Refined vertex
        vr, vc = find_vertex(pos, SEARCH_ROW, SEARCH_COL,
                             SEARCH_H, SEARCH_W, DENSITY_R)

        # Pixels inside the final evaluation box
        in_box = (
            (rows >= vr - BOX_H) & (rows < vr + BOX_H) &
            (cols >= vc - BOX_W) & (cols < vc + BOX_W)
        )

        # Draw
        ax.scatter(cols, rows, s=0.5, c="white", linewidths=0)
        ax.scatter(cols[in_box], rows[in_box], s=1.0, c="yellow", linewidths=0, zorder=3)

        # Search region (dashed blue)
        ax.add_patch(patches.Rectangle(
            (SEARCH_COL - SEARCH_W, SEARCH_ROW), 2 * SEARCH_W, SEARCH_H,
            linewidth=1.2, edgecolor="deepskyblue", facecolor="none",
            linestyle="--", zorder=4,
        ))
        # Evaluation box (solid red)
        ax.add_patch(patches.Rectangle(
            (vc - BOX_W, vr - BOX_H), 2 * BOX_W, 2 * BOX_H,
            linewidth=1.5, edgecolor="red", facecolor="none", zorder=5,
        ))
        # Vertex marker
        ax.scatter([vc], [vr], c="red", marker="+", s=100, zorder=6)

        ax.set_facecolor("black")
        ax.set_xlim(0, 500); ax.set_ylim(500, 0)
        ax.set_aspect("equal")
        ax.set_title(
            f"[{cls_name}] img {img_idx}\n"
            f"vertex=({vr:.0f},{vc:.0f})  in_box={in_box.sum()}/{len(pos)}",
            fontsize=8,
        )

    fig.suptitle(
        f"{cls_name}  |  search (dashed blue)  eval box (red)  r={DENSITY_R}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

## Pixel counts inside evaluation box per class

In [ ]:
N_EVAL = 300
rng2   = np.random.default_rng(1)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_imgs = np.where(labels == cls_idx)[0]
    if len(cls_imgs) == 0:
        continue
    sample = rng2.choice(cls_imgs, size=min(N_EVAL, len(cls_imgs)), replace=False)
    in_box = []
    for img_idx in sample:
        sl  = slice(offsets[img_idx], offsets[img_idx + 1])
        pos = positions[sl].astype(np.float32)
        rows, cols = pos[:, 0], pos[:, 1]
        vr, vc = find_vertex(pos, SEARCH_ROW, SEARCH_COL,
                             SEARCH_H, SEARCH_W, DENSITY_R)
        n = int((
            (rows >= vr - BOX_H) & (rows < vr + BOX_H) &
            (cols >= vc - BOX_W) & (cols < vc + BOX_W)
        ).sum())
        in_box.append(n)
    in_box = np.array(in_box)
    print(f"{cls_name:6s}  median={np.median(in_box):.0f}  mean={in_box.mean():.1f}  "
          f"min={in_box.min()}  max={in_box.max()}  zero_frac={(in_box==0).mean():.1%}")

## Chosen parameters → CLI flags

```bash
python -m dino.diagnostics.plot_knn features.npz \
    --vertex_knn \
    --search_row 0 --search_col 250 \
    --search_h SEARCH_H --search_w SEARCH_W \
    --density_r DENSITY_R \
    --box_h BOX_H --box_w BOX_W \
    --n_vertex_images 500
```